In [ ]:
import h5py
with h5py.File("./data/baryons_logPkR.h5", 'r') as f:
    print(f.keys())
    print(f['antilles'].keys())

In [2]:
import numpy as np
import scipy

nsrcs = 5
nlens = 10
ntheta = 20
ndv = (nsrcs*(nsrcs+1)+nsrcs*nlens+nlens)*ntheta

baseline  = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/lsst_real/chains/lsst_baseline_evaluate/lsst_baseline.modelvector')[:,1]
                        
pollution = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/lsst_real/chains/lsst_pollution_evaluate/lsst_pollution.modelvector')[:,1]
print(f'len of baseline {len(baseline)}; len of polution {len(pollution)}; expected ndv {ndv}')

cov_raw = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/lsst_real/data/cov_lsst')
cov = np.zeros((ndv,ndv))
for i in range(len(cov_raw)):
    ii,jj = int(cov_raw[i,0]),int(cov_raw[i,1])
    component = cov_raw[i,8]+cov_raw[i,9]
    cov[ii,jj] = component
    cov[jj,ii] = component

#mask = np.ones(ndv)
mask_full = np.loadtxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/lsst_real/data/lsst_full_shear_DESY3_lens.mask')[:,1]
tol = 0.5/20

def probechi2(start,end,probe):
    delta = (pollution - baseline)[start:end]
    mask = mask_full[start:end].astype(bool)
    delta_masked = delta[mask]
    cov_delta = cov[start:end,:][:,start:end]
    cov_delta_masked = cov_delta[mask,:][:,mask]
    invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
    chi2 = delta_masked@invcov_delta_masked@delta_masked
    print(f'chi2 of probe {probe} is {chi2}')
    print(f'{np.sum(mask)}/{len(mask)} points remain for probe {probe}')

ncombo0=0
cnt = 0
for ni in range(nsrcs):
    for nj in range(ni, nsrcs):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        idx_mask=0
        while(chi2>tol):
            mask_full[cnt*ntheta+idx_mask] = 0
            mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
            delta_masked = delta[mask]
            cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
            cov_delta_masked = cov_delta[mask,:][:,mask]
            invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
            chi2 = delta_masked@invcov_delta_masked@delta_masked
            idx_mask+=1
        print(f'xi+ ni={ni}, nj={nj}, chi2={chi2}')
        cnt+=1
ncombo1=cnt
probechi2(ncombo0*ntheta, ncombo1*ntheta, 'xi+')
for ni in range(nsrcs):
    for nj in range(ni, nsrcs):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        idx_mask=0
        while(chi2>tol):
            mask_full[cnt*ntheta+idx_mask] = 0
            mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
            delta_masked = delta[mask]
            cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
            cov_delta_masked = cov_delta[mask,:][:,mask]
            invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
            chi2 = delta_masked@invcov_delta_masked@delta_masked
            idx_mask+=1
        print(f'xi- ni={ni}, nj={nj}, chi2={chi2}')
        cnt+=1
ncombo2=cnt
probechi2(ncombo1*ntheta, ncombo2*ntheta,'xi-')
for ni in range(nlens):
    for nj in range(nsrcs):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        print(f'gammat ni={ni}, nj={nj}, chi2={chi2}')
        cnt+=1
ncombo3=cnt
probechi2(ncombo2*ntheta, ncombo3*ntheta,'gammat')
for ni in range(nlens):
        delta = (pollution - baseline)[cnt*ntheta:(cnt+1)*ntheta]
        mask = mask_full[cnt*ntheta:(cnt+1)*ntheta].astype(bool)
        delta_masked = delta[mask]
        cov_delta = cov[cnt*ntheta:(cnt+1)*ntheta,:][:,cnt*ntheta:(cnt+1)*ntheta]
        cov_delta_masked = cov_delta[mask,:][:,mask]
        invcov_delta_masked = scipy.linalg.inv(cov_delta_masked)
        chi2 = delta_masked@invcov_delta_masked@delta_masked
        print(f'wtheta ni={ni}, nj={ni}, chi2={chi2}')
        cnt+=1
ncombo4=cnt
probechi2(ncombo3*ntheta, ncombo4*ntheta,'wtheta')

len of baseline 1800; len of polution 1800; expected ndv 1800
xi+ ni=0, nj=0, chi2=0.006747338026568314
xi+ ni=0, nj=1, chi2=0.015119508771612698
xi+ ni=0, nj=2, chi2=0.014501088011089123
xi+ ni=0, nj=3, chi2=0.01212266679933766
xi+ ni=0, nj=4, chi2=0.017956365137143472
xi+ ni=1, nj=1, chi2=0.01959905724625981
xi+ ni=1, nj=2, chi2=0.006258515966106075
xi+ ni=1, nj=3, chi2=0.022577131296746836
xi+ ni=1, nj=4, chi2=0.01633238273780322
xi+ ni=2, nj=2, chi2=0.013454207917271168
xi+ ni=2, nj=3, chi2=0.01525675776827549
xi+ ni=2, nj=4, chi2=0.010811712462626463
xi+ ni=3, nj=3, chi2=0.006949330298254647
xi+ ni=3, nj=4, chi2=0.005973716656811812
xi+ ni=4, nj=4, chi2=0.01873184780820742
chi2 of probe xi+ is 0.0826520934896007
141/300 points remain for probe xi+
xi- ni=0, nj=0, chi2=0.017558606506895922
xi- ni=0, nj=1, chi2=0.01762675305111308
xi- ni=0, nj=2, chi2=0.012008188556305737
xi- ni=0, nj=3, chi2=0.008280759850027242
xi- ni=0, nj=4, chi2=0.015501151665071756
xi- ni=1, nj=1, chi2=0.01010

In [3]:
nlen = len(mask_full)
out = np.zeros((nlen,2))
out[:,0] = np.arange(nlen)
out[:,1] = mask_full
print(f'check the number: {np.sum(mask_full):.0f}/{len(mask_full)} of this mask')
np.savetxt('/project/chihway/junzhou/cocoa_approx/Cocoa/projects/lsst_real/data/lsst_Y3.mask', out)

check the number: 875/1800 of this mask
